# Certificate-forced induced subgraphs for the $K_4^3$ in $K_5^3$-free problem

This notebook verifies the following certificate-based implication.

If an exact certificate at truncation size $6$ proves the extremal value and a $6$-vertex graph has positive density in a hypothetical extremal sequence, then that $6$-vertex graph must be sharp. So it is enough to enumerate the sharp $6$-vertex graphs from the certificate, check that they all occur in the balanced bipartite construction, and then take induced closures to sizes $4$ and $5$.


In [1]:
from itertools import combinations
import pickle
from fractions import Fraction

from sage.all import QQ
from sage.algebras.combinatorial_theory import Theory

try:
    ThreeGraphTheory
except NameError:
    ThreeGraphTheory = Theory("ThreeGraph", arity=3)

TG = ThreeGraphTheory
TG.reset()
TG.printlevel(0)


def complete_3_edges(r):
    return [list(e) for e in combinations(range(r), 3)]


def complete_3graph(r):
    return TG(r, edges=complete_3_edges(r))


def to_sage(dim, data):
    if dim == 0:
        if isinstance(data, Fraction):
            return QQ(data)
        return data
    return [to_sage(dim - 1, x) for x in data]


def load_certificate(file_or_name):
    if not isinstance(file_or_name, str):
        return file_or_name
    fname = file_or_name if file_or_name.endswith(".pickle") else file_or_name + ".pickle"
    with open(fname, "rb") as f:
        return pickle.load(f)


def sharp_flags_from_certificate(theory, file_or_cert):
    cert = load_certificate(file_or_cert)
    N = cert["target size"]
    slacks = to_sage(1, cert["slack vector"])
    flagsN = list(theory.generate(N))
    sharp = [F for s, F in zip(slacks, flagsN) if s == 0]
    return cert, sharp


def positive_density_flags(theory, construction, k):
    return [F for F in theory.generate(k) if construction.density(F) != 0]


def flag_key(F):
    return str(F)


def unique_flags(flags):
    out = []
    seen = set()
    for F in flags:
        key = flag_key(F)
        if key not in seen:
            seen.add(key)
            out.append(F)
    return out


def induced_k_subflags(F, k):
    pts = range(F.size())
    subs = []
    for S in combinations(pts, k):
        subs.append(F.subflag(ftype_points=[], points=list(S)))
    return unique_flags(subs)


def closure_from_sharp_flags(sharp_flags, k):
    subs = []
    for F in sharp_flags:
        subs.extend(induced_k_subflags(F, k))
    return unique_flags(subs)


def print_flag_list(title, flags):
    print("=" * 80)
    print(title)
    print("=" * 80)
    for F in flags:
        print(F)
    print(f"count = {len(flags)}")
    print()


# ----------------------------------------------------------------------
# Problem setup
# ----------------------------------------------------------------------

K4 = complete_3graph(4)
K5 = complete_3graph(5)
BIPARTITE_TEMPLATE = [[0, 0, 1], [0, 1, 1]]

TG.exclude(K5)

N = 6
cert_name = "K4_in_K5_free_bipartite_forced_subgraphs_N6"
constr6 = TG.blowup_construction(N, 2, edges=BIPARTITE_TEMPLATE)

print("candidate lower bound =", constr6.density(K4))

bound6 = TG.optimize(
    K4,
    N,
    maximize=True,
    exact=True,
    construction=constr6,
    file=cert_name,
    denom=2**20,
    kernel_denom=2**20,
    slack_threshold=1e-8,
    printlevel=1,
)

print("certificate upper bound =", bound6)
TG.verify(cert_name)
print()

# ----------------------------------------------------------------------
# Step 1. Sharp 6-vertex graphs from the certificate
# ----------------------------------------------------------------------

cert, sharp6 = sharp_flags_from_certificate(TG, cert_name)
print_flag_list("Sharp induced 6-vertex graphs from the certificate", sharp6)

# Compare with the candidate construction
occurring6 = positive_density_flags(TG, constr6, 6)
occurring6_keys = {flag_key(F) for F in occurring6}
sharp6_in_candidate = [F for F in sharp6 if flag_key(F) in occurring6_keys]
sharp6_not_in_candidate = [F for F in sharp6 if flag_key(F) not in occurring6_keys]

print_flag_list("Sharp 6-vertex graphs that occur in the bipartite construction", sharp6_in_candidate)
print_flag_list("Sharp 6-vertex graphs that do NOT occur in the bipartite construction", sharp6_not_in_candidate)

if len(sharp6_not_in_candidate) == 0:
    print("PASS: every sharp 6-vertex graph appears in the bipartite construction.")
else:
    print("FAIL: there are sharp 6-vertex graphs outside the bipartite construction.")
print()

# ----------------------------------------------------------------------
# Step 2. Induced-closure on 4 and 5 vertices
# ----------------------------------------------------------------------

for k in [4, 5, 6]:
    if k == 6:
        forced_k = unique_flags(sharp6)
    else:
        forced_k = closure_from_sharp_flags(sharp6, k)

    candidate_k = positive_density_flags(TG, constr6, k)
    candidate_keys = {flag_key(F) for F in candidate_k}

    forced_in_candidate = [F for F in forced_k if flag_key(F) in candidate_keys]
    forced_not_in_candidate = [F for F in forced_k if flag_key(F) not in candidate_keys]

    print_flag_list(f"Forced induced {k}-vertex graphs coming from the sharp 6-graphs", forced_k)
    print_flag_list(f"Those forced induced {k}-vertex graphs that occur in the bipartite construction", forced_in_candidate)
    print_flag_list(f"Those forced induced {k}-vertex graphs that do NOT occur in the bipartite construction", forced_not_in_candidate)

    if len(forced_not_in_candidate) == 0:
        print(f"PASS: every induced {k}-vertex graph forced by the certificate occurs in the bipartite construction.")
    else:
        print(f"FAIL: some induced {k}-vertex graph forced by the certificate does not occur in the bipartite construction.")
    print()

# ----------------------------------------------------------------------
# Step 3. Optional equality check with the candidate lists
# ----------------------------------------------------------------------

for k in [4, 5, 6]:
    if k == 6:
        forced_k = unique_flags(sharp6)
    else:
        forced_k = closure_from_sharp_flags(sharp6, k)
    candidate_k = positive_density_flags(TG, constr6, k)

    forced_keys = {flag_key(F) for F in forced_k}
    candidate_keys = {flag_key(F) for F in candidate_k}

    missing_from_forced = [F for F in candidate_k if flag_key(F) not in forced_keys]

    print_flag_list(f"Candidate-induced {k}-vertex graphs not proved forced by this certificate", missing_from_forced)
    if len(missing_from_forced) == 0:
        print(f"The certificate gives equality for size {k}: forced list = candidate list.")
    else:
        print(f"For size {k}, the certificate only proves a subset relation, not equality.")
    print()


candidate lower bound = 3/8
Base flags generated, their number is 2102
The relevant ftypes are constructed, their number is 6
Block sizes before symmetric/asymmetric change is applied: [12, 64, 64, 64, 64, 63]


Done with mult table for Ftype on 4 points with edges=(012 013 023 123): : 6it [00:01,  5.47it/s]


Tables finished
Constraints finished
Adjusting table with kernels from construction
Running SDP after kernel correction. Used block sizes are [7, 3, 10, 53, 20, 44, 28, 36, 19, 44, 9, 51, -2102, -2]
CSDP 6.2.0
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 4.75e-01 Pobj: -4.3259900e+01 Ad: 3.05e-01 Dobj:  4.0189736e+01 
Iter:  2 Ap: 1.00e+00 Pobj: -8.8591319e+01 Ad: 7.93e-01 Dobj:  1.3679782e+01 
Iter:  3 Ap: 1.00e+00 Pobj: -9.3998594e+01 Ad: 8.93e-01 Dobj:  1.2809054e+00 
Iter:  4 Ap: 1.00e+00 Pobj: -9.8675030e+01 Ad: 9.44e-01 Dobj: -8.1442036e-02 
Iter:  5 Ap: 9.05e-01 Pobj: -1.0177451e+02 Ad: 8.60e-01 Dobj: -1.2710714e-01 
Iter:  6 Ap: 5.39e-01 Pobj: -1.0612390e+02 Ad: 8.71e-01 Dobj: -6.7428975e-02 
Iter:  7 Ap: 7.26e-01 Pobj: -9.3575790e+01 Ad: 7.02e-01 Dobj: -6.7542716e-02 
Iter:  8 Ap: 9.93e-01 Pobj: -4.1023646e+01 Ad: 8.16e-01 Dobj: -6.7884902e-02 
Iter:  9 Ap: 1.00e+00 Pobj: -7.6254722e+00 Ad: 8.36e-01 Dobj: -6.8393200e-02 
Iter: 10 A

100%|█████████████████████████████████████████████| 6/6 [02:07<00:00, 21.23s/it]


This took 127.39924383163452s
Final rounded bound is 3/8
certificate upper bound = 3/8
Checking X matrices


...

Solution matrices are all positive semidefinite, linear coefficients are all non-negative
Calculating multiplication tables


6it [00:16,  2.81s/it]


Done calculating linear constraints
Calculating the bound provided by the certificate


6it [01:19, 13.28s/it]


The solution is valid, it proves the bound 3/8

Sharp induced 6-vertex graphs from the certificate
Flag on 6 points, ftype from () with edges=()
Flag on 6 points, ftype from () with edges=(012 013 014 015 023 024 025 034 035 045)
Flag on 6 points, ftype from () with edges=(012 013 014 015 023 024 025 034 035 045 123 124 125 134 135 145)
Flag on 6 points, ftype from () with edges=(012 013 014 023 024 025 034 035 045 123 124 125 134 135 145 235 245 345)
count = 4

Sharp 6-vertex graphs that occur in the bipartite construction
Flag on 6 points, ftype from () with edges=()
Flag on 6 points, ftype from () with edges=(012 013 014 015 023 024 025 034 035 045)
Flag on 6 points, ftype from () with edges=(012 013 014 015 023 024 025 034 035 045 123 124 125 134 135 145)
Flag on 6 points, ftype from () with edges=(012 013 014 023 024 025 034 035 045 123 124 125 134 135 145 235 245 345)
count = 4

Sharp 6-vertex graphs that do NOT occur in the bipartite construction
count = 0

PASS: every sharp 6-v